In [67]:
import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

import math
from collections import Counter

In [8]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ESHAAN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ESHAAN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

# TOKENIZING


In [ ]:
class CaptionTokeniser :
    def __init__(self,  max_vocab_size = None):
        self.max_vocab_size = max_vocab_size
        
        self.word_to_idx = {
            "<PAD>": 0,
            "<BOS>": 1,
            "<EOS>": 2,
            "<UNK>": 3
        }


        self.vocab = self.word_to_idx
        self.idx_to_word = {
            0 : "<PAD>",
            1 : "<BOS>",
            2 : "<EOS>",
            3 : "<UNK>"
        }

    def create_vocab(self,text):
        tokens = word_tokenize(text.lower())
        tokens = [
            word for word in tokens
            if word.isalpha()
        ]
        word_counts = Counter(tokens)
        for word, count in word_counts.most_common():
            # if count >= 2:
            self.word_to_idx[word] = len(self.word_to_idx)
            self.idx_to_word[len(self.idx_to_word)] = word


    def add_to_vocab(self,text):
        words = [word for word in word_tokenize(text.lower())
                 if word.isalpha()]
        candidate_vocab = set(words)
        for word in candidate_vocab :
            if word not in self.vocab.keys():
                self.word_to_idx[word] = len(self.vocab.keys())
                self.idx_to_word[len(self.idx_to_word.keys())] = word


    def encode_arr(self, arr):
        out = [self.word_to_idx[word] for word in arr]
        return out

    def decode_arr(self, idx_arr):
        out = [self.idx_to_word[idx] for idx in idx_arr]
        return out

    

In [72]:
txt = ''
with open(r'C:\Users\ESHAAN\HAKUR\ML-CODE\datasets\hp_books\Book1.txt', 'r', encoding="utf-8") as f :
    txt += f.read()

In [ ]:
tokeniser = CaptionTokeniser()
tokeniser.create_vocab(txt)
tokeniser.idx_to_word

In [74]:
sentences = []
for i in sent_tokenize(txt):
    sent = []
    for word in word_tokenize(i):
        if word.isalpha():
           sent.append(word.lower())
    if len(sent) > 1 :
        sentences.append(sent)

In [75]:
sentences = [tokeniser.encode(sent) for sent in sentences]

In [ ]:
sentences[0] 

In [ ]:
tokeniser.decode(sentences[5])

# DATASETS and DATALOADER

In [99]:
class cutomDataset(Dataset):
    def __init__(self, sentences):
        
        super().__init__()
        self.features = []
        self.labels = []

        BOS = tokeniser.word_to_idx["<BOS>"]
        EOS = tokeniser.word_to_idx["<EOS>"]
        PAD = tokeniser.word_to_idx["<PAD>"]
        
        max_T = max(len(sentence) for sentence in sentences)


        for sentence in sentences:
            feature = [BOS] + sentence
            label = sentence + [EOS]
            padding_needed = max_T - len(sentence)
            feature += [PAD] * padding_needed
            label += [PAD] * padding_needed

            self.features.append(
                torch.tensor(feature, dtype=torch.long)
            )

            self.labels.append(
                torch.tensor(label, dtype=torch.long)
            )


    def __len__(self):
         return len(self.features)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.features[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )


dataset = cutomDataset(sentences[:64])
dataloader = DataLoader(dataset, batch_size=32)

# MODEL - SKELETON

In [109]:
class MultiHeadAttention(nn.Module):
    def __init__(self,embed_dim, num_heads, masked = False):
        super().__init__()
        if embed_dim%num_heads != 0 :
            raise ValueError(f"cant divide embed_dim{embed_dim} into {num_heads} heads")

        self.masked = masked
        self.num_heads = num_heads
        self.q = nn.Linear(embed_dim,embed_dim)
        self.k = nn.Linear(embed_dim,embed_dim)
        self.v = nn.Linear(embed_dim,embed_dim)
        self.Wo = nn.Linear(embed_dim,embed_dim)


    def forward(self, input_batch): 
        B, T, E = input_batch.shape
        device = input_batch.device
        H = self.num_heads
        Eh = E//self.num_heads

        query_vec = self.q(input_batch)  # shape (B,T,E)x(ExE) = BxTxE
        key_vec = self.k(input_batch)    # shape (B,T,E)x(ExE) = BxTxE
        value_vec = self.v(input_batch)  # shape (B,T,E)x(ExE) = BxTxE

        head_q_vecs = query_vec.reshape(B,T,H,Eh).transpose(1,2)    #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh) 
        head_k_vecs = key_vec.reshape(B,T,H,Eh).transpose(1,2)      #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh)
        head_v_vecs = value_vec.reshape(B,T,H,Eh).transpose(1,2)    #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh)

        sim_scores = head_q_vecs @ head_k_vecs.transpose(-2,- 1) # (B,H,T,Eh) . (B,H,Eh,T) = (B,H,T,T)
        sim_scores = sim_scores/math.sqrt(Eh) #(B,H,T,T)

        if self.masked :
            t_q = torch.arange(T, device=device).view(T,1)
            t_k = torch.arange(T, device=device).view(1,T)
            mask_0 = t_q >= t_k
            mask_inf = t_q < t_k 
            sim_scores = sim_scores.masked_fill(
                mask_inf,
                float('-inf')
            )         
        sim_scores = torch.softmax(sim_scores, dim = -1)#(B,H,Tq,Tk) , dim =1 , as we are softmaxing ALL KEY probs for a QUERY


        attention = sim_scores @ head_v_vecs # (B,H,T,T).(B,H,T,Eh) = (B,H,T,Eh)
        out = attention.transpose(1,2).reshape(B,T,E) # from trnaspose : (B,T,H,Eh), from reshape : (B,T,E)
        output = self.Wo(out) #(B,T,E)x(E,E) = (B,T,E)
        return output #(B,T,E)
        

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, masked = False):
        super().__init__()
        self.q_w = nn.Linear(embed_dim,embed_dim)
        self.k_w = nn.Linear(embed_dim,embed_dim)
        self.v_w = nn.Linear(embed_dim,embed_dim)
        self.o_w = nn.Linear(embed_dim,embed_dim)
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.masked = masked

        
    def forward(self, batch_dec, batch_enc):    
        
        B,T_dec,E = batch_dec.shape  #shape of both batches
        _,T_enc,_ = batch_enc.shape  #shape of both batches
        H = self.num_heads
        Eh = E//H

        if E!= self.embed_dim:
            raise ValueError(f'EMBED DIM MISMATCH during instantiation : {self.embed_dim}, given batch has embed dim of {E}')
        
        if E % H != 0:
            raise ValueError(f"can't divide embed_dim {E} into {H} heads")


        query_vecs = self.q_w(batch_dec) # B,T_dec,E
        key_vecs = self.k_w(batch_enc) # B,T_enc,E
        value_vecs = self.v_w(batch_enc) # B,T_enc,E

        query_vecs = query_vecs.reshape(B,T_dec,H,Eh).transpose(1,2) #from reshape: (B,T_dec,H,Eh) , from transpose : (B,H,T_dec,Eh)
        key_vecs = key_vecs.reshape(B,T_enc,H,Eh).transpose(1,2)    #from reshape: (B,T_enc,H,Eh) , from transpose : (B,H,T_enc,Eh)
        value_vecs = value_vecs.reshape(B,T_enc,H,Eh).transpose(1,2)    #from reshape: (B,T_enc,H,Eh) , from transpose : (B,H,T_enc,Eh)

        sim_scores = query_vecs @ key_vecs.transpose(-1,-2) # (B,H,T_dec,Eh) @ (B,H,Eh,T_enc) = (B,H,T_dec,T_enc)
        if self.masked:
            T_q = torch.arange(T_dec).view((T_dec,1))
            T_k = torch.arange(T_enc).view((1,T_enc))
            mask_inf = T_k > T_q

            sim_scores = sim_scores.masked_fill(
                mask_inf,
                float('-inf')
            )
        sim_scores = sim_scores/math.sqrt(Eh)
        sim_scores = torch.softmax(sim_scores, dim=-1)

        cross_attention = sim_scores @ value_vecs # (B,H,T_dec,T_enc) @ (B,H,T_enc,Eh) = (B,H,T_dec,Eh)
        cross_attention = cross_attention.transpose(1,2) # (B,T_dec,H,Eh)
        cross_attention = cross_attention.reshape(B,T_dec,E) # (B,T_dec,E)
        output = self.o_w(cross_attention) # (B,T_dec,E) @ (E,E) = (B,T_dec,E)

        return output


In [110]:
class ResidualConnect(nn.Module):
    def __init__(self, sublayer):
        super().__init__()
        self.sublayer = sublayer
        
    def forward(self,x):
        return x + self.sublayer(x)

In [111]:
def positionalEncoder(tensor):
    B,T,E = tensor.shape
    pos_encods = torch.zeros((1,T,E), device=tensor.device)
    for t in range(T) :
        for e in range(0,E,2) :
            pos_encods[:,t,e] = math.sin(t/math.pow(10000, e/E))
            pos_encods[:,t,e+1] = math.cos(t/math.pow(10000, e/E))
    return pos_encods


class AddPositionalEmbeds(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, tensor):
        B,T,E = tensor.shape
        pos_encods = positionalEncoder(tensor)
        return tensor + pos_encods
        
        

In [112]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, attention_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.MaskedAttention = MultiHeadAttention(embed_dim, attention_heads, masked=True)
        self.fnn = nn.Sequential(
            nn.Linear(embed_dim, 2048),
            nn.ReLU(),
            nn.Linear(2048,embed_dim)
        )
        self.layerNorm1 = nn.LayerNorm(self.embed_dim)
        self.layerNorm2 = nn.LayerNorm(self.embed_dim)

    def forward(self, input_embeds): 
        o = input_embeds + self.MaskedAttention(input_embeds) # (B,T,E)
        o = self.layerNorm1(o) # (B,T,E)
        o = o + self.fnn(o) # (B,T,E)
        o = self.layerNorm2(o) # (B,T,E)
        return o # (B,T,E)

In [113]:
class SimpleDecoder(nn.Module):
    def __init__(self, vocab_count, embedding_dim,attention_heads, n_blocks):
        super().__init__()
        self.n_blocks = n_blocks
        self.embedding = nn.Embedding(vocab_count, embedding_dim)
        self.add_positional = AddPositionalEmbeds()
        self.onn = nn.Sequential(
                    nn.Linear(embedding_dim, vocab_count), #(B,T,E)
                    # nn.Softmax(dim = -1)
        )

        self.decoders = nn.ModuleList(DecoderBlock(embedding_dim, attention_heads) for _ in range(n_blocks))

        

    def forward(self,batch):
        B,T = batch.shape
        embeds = self.embedding(batch) # shape = (B,T,E)
        embeds = self.add_positional(embeds)
        o = embeds # (B,T,E)
        for block in self.decoders :
            o = block(o) # (B,T,E)
        o = self.onn(o)
        return o
        
        

In [114]:
embedding_dim = 128
attention_heads = 4
n_blocks = 2

vocab_count = len(tokeniser.vocab)

In [115]:
model = SimpleDecoder(
    vocab_count=vocab_count,
    embedding_dim=embedding_dim,
    attention_heads=attention_heads,
    n_blocks=n_blocks
)

In [116]:
features, labels = next(iter(dataloader))

print("Features:", features.shape)
print("Labels:", labels.shape)

Features: torch.Size([32, 47])
Labels: torch.Size([32, 47])


C:\Users\ESHAAN\AppData\Local\Temp\ipykernel_35768\1349012235.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(self.features[idx], dtype=torch.long),
C:\Users\ESHAAN\AppData\Local\Temp\ipykernel_35768\1349012235.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(self.labels[idx], dtype=torch.long)


In [118]:
dataset = cutomDataset(sentences)

dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False
)

vocab_count = len(tokeniser.vocab)

model = SimpleDecoder(
    vocab_count=vocab_count,
    embedding_dim=128,
    attention_heads=4,
    n_blocks=2
)


features, labels = next(iter(dataloader))

print("Input:")
print(features.shape)
print(features.dtype)

print("\nLabels:")
print(labels.shape)
print(labels.dtype)


with torch.no_grad():

    output = model(features)

print("\nOutput:")
print(output.shape)


B, T = features.shape

assert output.shape == (
    B,
    T,
    vocab_count
)

assert labels.shape == (
    B,
    T
)

print("\nEverything passed!")

C:\Users\ESHAAN\AppData\Local\Temp\ipykernel_35768\1349012235.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(self.features[idx], dtype=torch.long),
C:\Users\ESHAAN\AppData\Local\Temp\ipykernel_35768\1349012235.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(self.labels[idx], dtype=torch.long)


Input:
torch.Size([32, 210])
torch.int64

Labels:
torch.Size([32, 210])
torch.int64

Output:
torch.Size([32, 210, 5638])

Everything passed!


In [144]:
from transformers import AutoFeatureExtractor, ResNetForImageClassification
import torch
from datasets import load_dataset
from torchinfo import summary
from PIL import Image

dataset = load_dataset("huggingface/cats-image")
image = dataset["test"]["image"][0]

feature_extractor = AutoFeatureExtractor.from_pretrained("microsoft/resnet-34")
model = ResNetForImageClassification.from_pretrained("microsoft/resnet-34")
inputs = feature_extractor(image, return_tensors="pt")

spatial_feature_encoder = nn.Sequential(
    *list(model.children())[:-1]
)

test_tensor = torch.rand(10,3,244,244)
with torch.no_grad():
    y = spatial_feature_encoder(test_tensor).last_hidden_state
print(y.shape)


c:\Users\ESHAAN\HAKUR\ML-CODE\.venv\Lib\site-packages\transformers\models\convnext\feature_extraction_convnext.py:30: FutureWarning: The class ConvNextFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ConvNextImageProcessor instead.
  warnings.warn(


torch.Size([10, 512, 8, 8])


# IMAGE DATASET - DATALOADER

In [158]:
from transformers import AutoFeatureExtractor, ResNetForImageClassification
import torch
from datasets import load_dataset

dataset = load_dataset("huggingface/cats-image")
image = dataset["test"]["image"][0]

feature_extractor = AutoFeatureExtractor.from_pretrained("microsoft/resnet-34")
model = ResNetForImageClassification.from_pretrained("microsoft/resnet-34")

inputs = feature_extractor(image, return_tensors="pt")

with torch.no_grad():
    logits = model(**inputs).logits

# model predicts one of the 1000 ImageNet classes
predicted_label = logits.argmax(-1).item()
print(model.config.id2label[predicted_label])


c:\Users\ESHAAN\HAKUR\ML-CODE\.venv\Lib\site-packages\transformers\models\convnext\feature_extraction_convnext.py:30: FutureWarning: The class ConvNextFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ConvNextImageProcessor instead.
  warnings.warn(


tabby, tabby cat


In [ ]:

from pathlib import Path


txt = '../datasets/Captioning/captions.txt'
lines = []
with open(txt,'r') as file :
    lines = [line.split(',') for line in file.readlines()]
    lines = [ [ line[:-1][0], line[-1].replace('\n','') ] for line in lines]
# captions = lines[:,1]

lines.pop(0)
corpus = ''
for line in lines :
    # print(line[1])
    corpus += " "
    corpus+=str(line[1])



caption_tokeniser = CaptionTokeniser()
caption_tokeniser.create_vocab(corpus)

caption_dict = {}

for line in lines : 
    jpg_name = line[0]
    caption = str(line[1])
    sent = []
    for word in word_tokenize(caption.lower()):
        sent.append(word) if word.isalpha() else None
    encoded_caption = caption_tokeniser.encode(sent)
    if jpg_name in caption_dict:
        caption_dict[jpg_name].append(encoded_caption)
    else:
        caption_dict[jpg_name] = [encoded_caption]



In [163]:
image_dir = Path('../datasets/Captioning/Images')
jpegs = [f.name for f in image_dir.glob('*.jpg')]

class FeatureDataset(Dataset):
    def __init__(self, image_dir, jpg_list, caption_dict ):
        super().__init__()

        self.samples = []
        self.image_dir = image_dir
        for jpg_name in jpg_list:
            
            captions = caption_dict[jpg_name]
            
            for caption in captions :
                self.samples.append(
                    (jpg_name, caption)
                )
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        jpg_name, caption = self.samples[idx]
        jpg_path = self.image_dir/jpg_name
        jpg_image = Image.open(jpg_path).convert('RGB')
        with torch.no_grad():
            features = feature_extractor(jpg_image, return_tensors="pt").pixel_values.squeeze(0)

        caption = torch.tensor(caption , dtype=torch.long)
        return features , caption

dataset = FeatureDataset(image_dir, jpegs,caption_dict )

In [166]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images, captions = zip(*batch)

    images = torch.stack(images)

    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=caption_tokeniser.word_to_idx['<PAD>']
    )

    return images, captions

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

# RESNET - ENCODER

In [188]:
class SpatialEncoder(nn.Module):
    def __init__(self, device):
        super().__init__()

        model = ResNetForImageClassification.from_pretrained("microsoft/resnet-34")

        self.spatial_feature_encoder = nn.Sequential(
            *list(model.children())[:-1]
        )

    def forward(self, input):
        B,C,W,H = input.shape
        y = self.spatial_feature_encoder(input).last_hidden_state # (B,512,7,7) 
        B,C_o,W_o,H_o = y.shape
        y = y.transpose(1,3)
        y = y.reshape(B,W_o*H_o,C_o)

        return y
        


In [189]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SPE = SpatialEncoder(device=device)
f,l = next(iter(loader))
print(f.shape)
y = SPE(f)
print(y.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32, 49, 512])
